# mmBERTによる日本語文章の2値分類

`texts` と `labels`（0または1）を実データに置き換えて使う。モデル: [jhu-clsp/mmBERT-base](https://huggingface.co/jhu-clsp/mmBERT-base)

In [4]:
#!pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 5.4 MB/s eta 0:00:00


In [8]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_name = "jhu-clsp/mmBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
texts = [
    "この商品には満足しています。", "とても使いやすくて便利です。",
    "期待どおりの品質でした。", "また購入したいと思います。",
    "品質が悪くて残念です。", "まったく役に立ちません。",
    "すぐに壊れてしまいました。", "二度と購入しません。",
]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

dataset = Dataset.from_dict({"text": texts, "label": labels}).train_test_split(
    test_size=0.25, seed=42#, stratify_by_column="label"
)
dataset = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=256), batched=True
)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [10]:
args = TrainingArguments(
    output_dir="./mmbert-ja-binary",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=dataset["train"], eval_dataset=dataset["test"],
    processing_class=tokenizer,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss
1,1.480735,8.142886
2,1.117869,4.555985
3,0.550583,4.141571
4,0.082062,4.602029
5,0.000010,4.985686
6,0.000006,5.235787
7,0.000009,5.390997
8,0.000009,5.483926
9,0.000006,5.533604
10,0.000005,5.551645


Training Loss,Validation Loss,Epoch
0.000005,5.551645,10


{'eval_loss': 5.551645278930664}

In [23]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        probabilities = model(**inputs).logits.softmax(dim=-1)[0]
    label = int(probabilities.argmax())
    return {"label": label, "probability": float(probabilities[label])}

print(predict("操作が簡単で、とても気に入りました。"))
print(predict("とても便利です"))
print(predict("操作が面倒で、諦めた。"))
print(predict("こんなもの、誰が欲しがるのか、全く分からない"))

{'label': 1, 'probability': 0.9993826150894165}
{'label': 1, 'probability': 0.9986750483512878}
{'label': 0, 'probability': 0.9795970320701599}
{'label': 0, 'probability': 0.9710573554039001}
